# BarbellOS: ilustracje techniki z pozy manekina (Qwen-Image-Edit)

Gemini nie trzyma się techniki, więc pozę i sprzęt dyktuje kod: `scripts/technique_pose.py` rysuje szare manekiny (start i koniec ruchu, stopy w tym samym miejscu). Model **Qwen-Image-Edit-2511** (otwarty, Apache 2.0) maluje na nich naszą postać z `postac.png`.

**Zanim uruchomisz:** **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → L4** (albo A100, NIE T4) i włącz **Duża ilość pamięci RAM**. Potem **Środowisko wykonawcze → Uruchom wszystko**.

Pierwsze uruchomienie pobiera model (ok. 58 GB, przy szybkim łączu Colaba kilka minut). Na końcu pobierze się `wyniki.zip`.

## 1. Ćwiczenia i ustawienia

In [ ]:
# Ćwiczenia do wygenerowania (szkice muszą być w scripts/colab/szkice w repozytorium).
EXERCISES = ["martwy", "rdl"]
# Szybkie wersje z przyspieszaczem Lightning (4 kroki, ok. pół minuty na obrazek).
LIGHTNING_SEEDS = [1, 2, 3, 4]
# Pełne wersje bez przyspieszacza (40 kroków, ok. 10 minut na obrazek na L4). Próba 28.11.2026:
# jakość jak w Lightning, więc domyślnie wyłączone; wpisz np. [1], żeby porównać.
FULL_SEEDS = []
FULL_STEPS = 40

## 2. Karta graficzna

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil, torch
print(f"RAM: {psutil.virtual_memory().total / 1e9:.0f} GB")
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if gpu_gb < 20 or not torch.cuda.is_bf16_supported():
    raise SystemExit(
        "Ta karta jest za słaba (np. T4). Środowisko wykonawcze → Zmień typ środowiska wykonawczego "
        "→ L4 albo A100, potem Uruchom wszystko."
    )

## 3. Instalacja (diffusers z gałęzi głównej, jak każe karta modelu)

In [ ]:
!pip -q install -U git+https://github.com/huggingface/diffusers transformers accelerate bitsandbytes

## 4. Postać i szkice z repozytorium BarbellOS

Obrazek 1 to postać i styl, obrazek 2 to manekin w pozie.

In [ ]:
from diffusers.utils import load_image
from PIL import Image

BASE = "https://raw.githubusercontent.com/Preeksik/BarbellOS/main/scripts/colab/szkice"
character = load_image(f"{BASE}/postac.png").convert("RGB")
sketches = {
    name: [load_image(f"{BASE}/{name}-{i}.png").convert("RGB") for i in (1, 2)] for name in EXERCISES
}

def row(images, h=320):
    # Obrazki obok siebie, do podglądu.
    scaled = [im.resize((round(im.width * h / im.height), h)) for im in images]
    out = Image.new("RGB", (sum(im.width for im in scaled), h), "white")
    x = 0
    for im in scaled:
        out.paste(im, (x, 0))
        x += im.width
    return out

display(row([character] + [im for pair in sketches.values() for im in pair]))

## 5. Model: Qwen-Image-Edit-2511

Część rysująca (20 mld parametrów) jest skwantyzowana do 4 bitów (NF4), żeby zmieściła się na L4; części modelu idą na kartę na czas liczenia (`enable_model_cpu_offload`). Zajmie ok. 37 GB RAM.

Warstwa modulacji pierwszego bloku i warstwy wyjściowe zostają bez kwantyzacji (jak w przykładzie `diffusers` dla Qwen-Image NF4). Pierwsza próba 28.11.2026 kwantyzowała wszystko i dawała sam szum.

Przyspieszacz **Lightning** (lightx2v, LoRA na 4 kroki) ma własny harmonogram kroków; włączamy go tylko na czas szybkich wersji.

In [ ]:
import gc
from diffusers import QwenImageEditPlusPipeline
from diffusers.quantizers import PipelineQuantizationConfig

quant = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": torch.bfloat16,
        # Bez tego obraz to sam szum (próba 28.11.2026).
        "llm_int8_skip_modules": ["transformer_blocks.0.img_mod", "norm_out", "proj_out"],
    },
    components_to_quantize=["transformer"],
)
pipe = QwenImageEditPlusPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit-2511", quantization_config=quant, torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=None)
base_scheduler = pipe.scheduler

import math
from diffusers import FlowMatchEulerDiscreteScheduler

lightning_scheduler = FlowMatchEulerDiscreteScheduler.from_config({
    "base_image_seq_len": 256, "base_shift": math.log(3), "invert_sigmas": False,
    "max_image_seq_len": 8192, "max_shift": math.log(3), "num_train_timesteps": 1000, "shift": 1.0,
    "shift_terminal": None, "stochastic_sampling": False, "time_shift_type": "exponential",
    "use_beta_sigmas": False, "use_dynamic_shifting": True, "use_exponential_sigmas": False,
    "use_karras_sigmas": False,
})
try:
    pipe.load_lora_weights(
        "lightx2v/Qwen-Image-Edit-2511-Lightning",
        weight_name="Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors",
        adapter_name="lightning",
    )
    has_lightning = True
except Exception as error:
    print("Lightning się nie wczytał, zostają pełne wersje:", error)
    has_lightning = False
gc.collect()
print(f"RAM zajęty: {psutil.virtual_memory().used / 1e9:.0f} GB, model gotowy")

## 6. Opisy ćwiczeń

Model dostaje pozę z manekina, a w opisie to, czego manekin nie mówi: styl, mięśnie na niebiesko i najważniejsze punkty techniki.

In [ ]:
STYLE = (
    "same face and hair, anatomical ecorche muscle illustration style with grey muscles, black shorts, "
    "black shoes, pure white background, soft studio light, strict side profile view, he faces left"
)
SPEC = {
    "martwy": {
        "muscles": "glutes, hamstrings, lower back erectors and upper trapezius",
        "start": "conventional deadlift start: the two black plates REST ON THE FLOOR, the bar is over the middle "
                 "of his foot and touches his shins, the near plate hides his shins, straight arms, hips clearly "
                 "HIGHER than the knees (not a squat), shoulders slightly in front of the bar, back flat",
        "end": "deadlift lockout seen from the side: standing fully upright, hips and knees extended, holding the "
               "barbell in straight arms hanging down, bar at the top of the thighs, the near plate covers his "
               "thighs, no leaning back, the bar is NOT overhead",
    },
    "rdl": {
        "muscles": "glutes, hamstrings and lower back erectors",
        "start": "Romanian deadlift top seen from the side: standing upright, holding the barbell in straight "
                 "arms hanging down at the top of the thighs, knees slightly bent, the bar is NOT overhead",
        "end": "Romanian deadlift bottom: hips pushed far back, knees slightly bent, back flat, bar at mid-shin, "
               "plates NOT touching the floor",
    },
}
NEGATIVE = (
    "grey mannequin, cartoon shapes, extra limbs, extra fingers, plates floating above the floor, bar in front "
    "of the feet, rounded back, text, numbers, logos, watermark, floor line, blurry, low quality"
)

def pose_prompt(spec, key):
    return (
        f"Draw the man from image 1 ({STYLE}) in EXACTLY the pose, position and size of the grey mannequin in "
        f"image 2: same joint angles, same feet position on the floor, same barbell and plates position. "
        f"{spec[key]}. Muscles in blue (#2F6FD6) ONLY on {spec['muscles']}; all other muscles grey. "
        f"No text, no logos. The result must not look like a mannequin."
    )

## 7. Generowanie

Dla każdego ziarna oba obrazki (start i koniec ruchu) powstają tak samo: postać + manekin, to samo ziarno. Próba 28.11.2026 robiła koniec ruchu jako edycję gotowego startu i model ignorował manekina (wyciskanie nad głowę, kopia startu). Czas obrazka widać pod paskiem postępu.

In [ ]:
import os, shutil, time

# Czyste wyniki przy każdym uruchomieniu (inaczej do paczki trafiają pliki z poprzedniej próby).
shutil.rmtree("/content/wyniki", ignore_errors=True)
os.makedirs("/content/wyniki", exist_ok=True)

def edit(images, prompt, seed, fast):
    t = time.time()
    # Lightning: 4 kroki bez negatywnego opisu (jeden przebieg na krok). Pełny: 40 kroków, CFG 4.
    extra = (
        {"num_inference_steps": 4, "true_cfg_scale": 1.0}
        if fast
        else {"num_inference_steps": FULL_STEPS, "true_cfg_scale": 4.0, "negative_prompt": NEGATIVE}
    )
    with torch.inference_mode():
        image = pipe(
            image=images, prompt=prompt, height=1024, width=1024,
            generator=torch.Generator(device="cpu").manual_seed(seed), **extra,
        ).images[0]
    print(f"  {time.time() - t:.0f} s")
    return image

def run(mode, seeds):
    fast = mode == "lightning"
    if fast:
        pipe.enable_lora()
        pipe.scheduler = lightning_scheduler
    elif has_lightning:
        pipe.disable_lora()
        pipe.scheduler = base_scheduler
    for name in EXERCISES:
        spec, (sketch_start, sketch_end) = SPEC[name], sketches[name]
        for seed in seeds:
            print(name, mode, "ziarno", seed, "start")
            start = edit([character, sketch_start], pose_prompt(spec, "start"), seed, fast)
            print(name, mode, "ziarno", seed, "koniec ruchu")
            end = edit([character, sketch_end], pose_prompt(spec, "end"), seed, fast)
            start.save(f"/content/wyniki/{name}-{mode}-s{seed}-1.png")
            end.save(f"/content/wyniki/{name}-{mode}-s{seed}-2.png")
            display(row([sketch_start, start, sketch_end, end]))

if has_lightning:
    run("lightning", LIGHTNING_SEEDS)
if FULL_SEEDS or not has_lightning:
    run("pelny", FULL_SEEDS or [1])

## 8. Pobranie

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/wyniki", "zip", "/content/wyniki")
files.download("/content/wyniki.zip")

## Co sprawdzić

- Czy **talerze stoją na podłodze** na starcie martwego, a gryf jest przy piszczelach.
- Czy **stopy są w tym samym miejscu** na obu obrazkach.
- Czy to wciąż **nasza postać** (styl jak w `postac.png`), a nie szary manekin.
- Czy niebieskie są tylko właściwe mięśnie.

Rozpakuj `wyniki.zip` do `exports\technika\qwen\` i napisz agentowi, które ziarno lepsze (albo niech oceni sam).